# 📄 Konversi CSV ke XLSX — Balanced Sample

> Notebook ini menyiapkan dataset sampling seimbang dari `cleandata.csv` lalu menyimpannya ke format Excel (`.xlsx`) untuk kebutuhan analisis lanjutan / anotasi manual.

---

## 🎯 Tujuan

Membuat dataset berukuran **500 baris** dengan komposisi seimbang **100 baris per rating (1-5)**, dengan syarat review memiliki minimal 2 kata agar kualitas teks tetap terjaga.

---

## 📋 Pipeline Overview

| # | Langkah | Input | Output |
|:--|:--------|:------|:-------|
| 1 | Load Data | `cleandata.csv` | `df` |
| 2 | Filter Minimum Kata | `content` | `df_filtered` |
| 3 | Balanced Sampling | `df_filtered` | `df_500` (100/rating) |
| 4 | Export XLSX | `df_500` | `df_honest_reviews_500.xlsx` |
| 5 | Verifikasi Output | file XLSX | ringkasan & preview |

## 📦 Install & Import Libraries

Library utama yang dibutuhkan:
- `pandas` untuk manipulasi data tabular
- `openpyxl` sebagai engine penulisan file Excel (`.xlsx`)

In [18]:
import pandas as pd

# openpyxl akan digunakan oleh pandas saat ekspor ke XLSX
print("✅ Libraries loaded")
print("📌 pandas version:", pd.__version__)

✅ Libraries loaded
📌 pandas version: 3.0.1


---
## 📂 Step 1 — Load Data CSV

Membaca file `cleandata.csv` hasil preprocessing dari notebook sebelumnya.

In [27]:
# Membaca file CSV
df = pd.read_csv('cleandata.csv')

print('=' * 55)
print('RINGKASAN DATA INPUT')
print('=' * 55)
print(f"📊 Total baris : {len(df):,}")
print(f"📊 Total kolom : {len(df.columns)}")
print(f"🧱 Nama kolom  : {df.columns.tolist()}")
print('=' * 55)

# Tampilkan teks review tanpa terpotong
pd.set_option('display.max_colwidth', None)
print("\n👁️ Preview 5 baris pertama (full text):")
display(df[['score', 'content', 'text_final', 'reviewCreatedVersion']].head())
pd.reset_option('display.max_colwidth')

RINGKASAN DATA INPUT
📊 Total baris : 39,164
📊 Total kolom : 4
🧱 Nama kolom  : ['score', 'content', 'text_final', 'reviewCreatedVersion']

👁️ Preview 5 baris pertama (full text):


,score,content,text_final,reviewCreatedVersion
0,5,ini adalah pertama kali saya menggunakan kartu kredit. yang katanya cocok buat anak muda. nanti saya update pengalaman saya setelah pakai kartu ini. terimakasih buat kak rima yang udah membimbing dengan sangat baik,kali kartu kredit cocok anak muda update alam pakai kartu terima kasih bimbing sangat baik,3.831.0
1,5,"pengajuan berhasil, pelayanan utk CS atas nama Rio sangat sopan dan penjelasan sangat terperinci, terima kasih Honest",hasil layan cs nama sangat sopan jelas sangat perinci terima kasih,3.833.1
2,5,Pengajuan cepat dan aman..,cepat aman,3.833.1
3,5,cs Ridwan terbaik,cs baik,3.833.1
4,5,penjelasan yang baik oleh mba rima,jelas baik,3.833.1


---
## ✂️ Step 2 — Filter & Balanced Sampling

Melakukan dua proses utama:
1. Filter review dengan minimal **2 kata** (`word_count > 1`)
2. Sampling seimbang **100 review per rating** (1 sampai 5)

In [28]:
# 1) Filter review minimal 2 kata
df['word_count'] = df['content'].fillna('').str.split().str.len()
df_filtered = df[df['word_count'] > 1].copy()

print(f"📊 Total baris setelah filter minimal 2 kata: {len(df_filtered):,}")
print("\n📊 Distribusi rating sebelum sampling:")
print(df_filtered['score'].value_counts().sort_index())

# 2) Sampling seimbang 100 baris per rating (1-5)
df_sampled_list = []
for rating in [1, 2, 3, 4, 5]:
    df_rating = df_filtered[df_filtered['score'] == rating]
    if len(df_rating) >= 100:
        df_sample = df_rating.sample(n=100, random_state=42)
    else:
        df_sample = df_rating
        print(f"⚠️ Warning: Rating {rating} hanya memiliki {len(df_sample)} baris")
    df_sampled_list.append(df_sample)

# Gabungkan hasil sampling
df_500 = pd.concat(df_sampled_list, ignore_index=True)

# Hapus kolom bantu
df_500 = df_500.drop('word_count', axis=1)

print("\n" + '=' * 55)
print('RINGKASAN SAMPLING')
print('=' * 55)
print(f"📦 Total baris hasil sampling : {len(df_500):,}")
print("📊 Distribusi rating akhir:")
print(df_500['score'].value_counts().sort_index())
print('=' * 55)

# Tampilkan teks review tanpa terpotong
pd.set_option('display.max_colwidth', None)
print("\n👁️ Preview 10 baris hasil sampling (full text):")
display(df_500[['score', 'content', 'text_final', 'reviewCreatedVersion']].head(10))
pd.reset_option('display.max_colwidth')

📊 Total baris setelah filter minimal 2 kata: 38,373

📊 Distribusi rating sebelum sampling:
score
1     2935
2      462
3      556
4      698
5    33722
Name: count, dtype: int64

RINGKASAN SAMPLING
📦 Total baris hasil sampling : 500
📊 Distribusi rating akhir:
score
1    100
2    100
3    100
4    100
5    100
Name: count, dtype: int64

👁️ Preview 10 baris hasil sampling (full text):


,score,content,text_final,reviewCreatedVersion
0,1,"Sempet happy dapat info aplikasi ini. Pengen dimudahkan untuk transaksi luar negri. Ternyata malah antiklimaks. Verifikasi wajah selalu bermasalah dgn alasan ""cari tempat terang dan jangan pakai aksesoris"". Padahal pake lampu penerangan sm coba di siang hari bolong, sama aja. Maap ya. Kecewa. 👎",happy info ken mudah transaksi negri verifikasi wajah masalah alas cari terang jangan pakai aksesoris pake lampu terang coba siang aja maap kecewa,2.843.4
1,1,Nunggu kode otp tidak ada juga,tunggu kode otp tidak,NaN
2,1,Jangan sampai data saya d salah gunakan ...soalnya tidak d setujui,jangan data salah tidak tuju,2.972.0
3,1,Kartunya belom dikirim sampai skrg janjinya ada kartunya,kartu belum kirim janji kartu,NaN
4,1,Sudah di acc tp kartu tidak dikirim jg ke alamat sudah lebih dr sebulan dari semenjak acc jadinya tidak bisa aktivasi aplikasinya...di wa ke operatornya jg jawabannya tidak jelas terus gak ada kepastian...!!,acc kartu tidak kirim alamat bulan semenjak acc tidak aktivasi wa operator jawab tidak jelas tidak pasti,NaN
5,1,"Sangat kecewa,proses nya lama bgt,kaga ada kabar nya,di suruh tunggu terus besok siang,udah besok siang suruh tunggu lg besok siang,kata dapat wa blast mudah pengajuan nya.tp sampai apk ini saya uninstall aja kaga ada tuh kabar nya,mohon feedback nya.",sangat kecewa proses lama banget kaga kabar suruh tunggu besok siang besok siang suruh tunggu besok siang wa mudah apk uninstall aja kaga tuh kabar mohon,NaN
6,1,Kurang di percaya,percaya,NaN
7,1,"Aneh baru kali ini ngajuin kartu kredit harus bayar biaya pembukaan akun 500rb kalau belum membayar, kartu tidak bisa aktif. mana ada kartu kredit belum dipakai harus bayar dulu. limitnya jg kecil. mikir lagi sebelum lanjut",aneh kali ngajuin kartu kredit bayar biaya buka akun belum bayar kartu tidak aktif kartu kredit belum pakai bayar limit mikir,3.698.0
8,1,auto unistall apk setiap mau masuk permuk Mulu saya tegaskan saya tidak bertangung jawab atas segala yg masuk di akun saya pertngl 12 Juni saya hps apk nya,auto unistall apk masuk mulu tidak masuk akun juni hps apk,NaN
9,1,kalo pengajuan berikutnya diterima baru sy kasih full stars,terima kasih full,NaN


---
## 💾 Step 3 — Export ke XLSX

Menyimpan hasil sampling (`df_500`) ke file Excel agar mudah dibuka dan dibagikan.

In [25]:
# Simpan ke format XLSX
output_file = 'df_honest_reviews_500.xlsx'
df_500.to_excel(output_file, index=False, engine='openpyxl')

print('=' * 55)
print('EKSPOR FILE')
print('=' * 55)
print(f"✅ File berhasil disimpan: {output_file}")
print(f"📊 Jumlah baris diekspor : {len(df_500):,}")
print(f"📊 Jumlah kolom diekspor : {len(df_500.columns)}")
print('=' * 55)

EKSPOR FILE
✅ File berhasil disimpan: df_honest_reviews_500.xlsx
📊 Jumlah baris diekspor : 500
📊 Jumlah kolom diekspor : 4


---
## ✅ Step 4 — Verifikasi Output

Membaca kembali file XLSX untuk memastikan file berhasil dibuat dan strukturnya sesuai.

In [29]:
# Membaca kembali file XLSX untuk verifikasi
df_verify = pd.read_excel(output_file)

print('=' * 55)
print('VERIFIKASI FILE XLSX')
print('=' * 55)
print(f"📄 Nama file    : {output_file}")
print(f"📊 Jumlah baris : {len(df_verify):,}")
print(f"📊 Jumlah kolom : {len(df_verify.columns)}")
print(f"🧱 Nama kolom   : {df_verify.columns.tolist()}")
print("\n📊 Distribusi rating:")
print(df_verify['score'].value_counts().sort_index())
print('=' * 55)

# Tampilkan teks review tanpa terpotong
pd.set_option('display.max_colwidth', None)
print("\n👁️ Preview 5 baris pertama (full text):")
display(df_verify[['score', 'content', 'text_final', 'reviewCreatedVersion']].head())
pd.reset_option('display.max_colwidth')

VERIFIKASI FILE XLSX
📄 Nama file    : df_honest_reviews_500.xlsx
📊 Jumlah baris : 500
📊 Jumlah kolom : 4
🧱 Nama kolom   : ['score', 'content', 'text_final', 'reviewCreatedVersion']

📊 Distribusi rating:
score
1    100
2    100
3    100
4    100
5    100
Name: count, dtype: int64

👁️ Preview 5 baris pertama (full text):


,score,content,text_final,reviewCreatedVersion
0,1,"Sempet happy dapat info aplikasi ini. Pengen dimudahkan untuk transaksi luar negri. Ternyata malah antiklimaks. Verifikasi wajah selalu bermasalah dgn alasan ""cari tempat terang dan jangan pakai aksesoris"". Padahal pake lampu penerangan sm coba di siang hari bolong, sama aja. Maap ya. Kecewa. 👎",happy info ken mudah transaksi negri verifikasi wajah masalah alas cari terang jangan pakai aksesoris pake lampu terang coba siang aja maap kecewa,2.843.4
1,1,Nunggu kode otp tidak ada juga,tunggu kode otp tidak,NaN
2,1,Jangan sampai data saya d salah gunakan ...soalnya tidak d setujui,jangan data salah tidak tuju,2.972.0
3,1,Kartunya belom dikirim sampai skrg janjinya ada kartunya,kartu belum kirim janji kartu,NaN
4,1,Sudah di acc tp kartu tidak dikirim jg ke alamat sudah lebih dr sebulan dari semenjak acc jadinya tidak bisa aktivasi aplikasinya...di wa ke operatornya jg jawabannya tidak jelas terus gak ada kepastian...!!,acc kartu tidak kirim alamat bulan semenjak acc tidak aktivasi wa operator jawab tidak jelas tidak pasti,NaN
